In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

# Загрузка данных
data = pd.read_csv('Metro_Interstate_Traffic_Volume.csv')

# Оставляем только нужные столбцы
data = data[['date_time', 'traffic_volume', 'holiday', 'temp']]

# Преобразование в datetime и установка в качестве индекса
data['date_time'] = pd.to_datetime(data['date_time'])
data = data.sort_values('date_time').set_index('date_time')

In [80]:
data

,traffic_volume,holiday,temp
date_time,,,
2012-10-02 09:00:00,5545,NaN,288.28
2012-10-02 10:00:00,4516,NaN,289.36
2012-10-02 11:00:00,4767,NaN,289.58
2012-10-02 12:00:00,5026,NaN,290.13
2012-10-02 13:00:00,4918,NaN,291.14
...,...,...,...
2018-09-30 19:00:00,3543,NaN,283.45
2018-09-30 20:00:00,2781,NaN,282.76
2018-09-30 21:00:00,2159,NaN,282.73


In [81]:
data['holiday'].unique()

<ArrowStringArray>
[                        nan,              'Columbus Day',
              'Veterans Day',          'Thanksgiving Day',
             'Christmas Day',             'New Years Day',
      'Washingtons Birthday',              'Memorial Day',
          'Independence Day',                'State Fair',
                 'Labor Day', 'Martin Luther King Jr Day']
Length: 12, dtype: str

In [82]:
# Если значение не NaN (т.е. там есть название праздника), ставим 1, иначе 0
data['is_holiday'] = data['holiday'].notna().astype(int)

In [83]:
# Размечаем праздник на весь день (если хоть один час помечен как праздник)
data['is_holiday'] = data.groupby(data.index.date)['is_holiday'].transform('max')

In [84]:
# Удаляем исходный столбец с названиями
data = data.drop(columns=['holiday'])

In [85]:
data[data['is_holiday'] == 1]

,traffic_volume,temp,is_holiday
date_time,,,
2012-10-08 00:00:00,455,273.08,1
2012-10-08 01:00:00,336,272.62,1
2012-10-08 02:00:00,265,271.78,1
2012-10-08 03:00:00,314,271.05,1
2012-10-08 04:00:00,779,270.63,1
...,...,...,...
2018-09-03 21:00:00,2081,294.80,1
2018-09-03 22:00:00,1588,294.49,1
2018-09-03 22:00:00,1588,294.49,1


In [87]:
# 1. Удаление дубликатов по индексу (времени)
data = data[~data.index.duplicated(keep='first')]

# 2. Выравнивание временных интервалов (каждый час)
full_range = pd.date_range(start=data.index.min(), end=data.index.max(), freq='h')
data = data.reindex(full_range)

# 3. Интерполяция пропусков для числовых переменных
data['traffic_volume'] = data['traffic_volume'].interpolate(method='linear')
data['temp'] = data['temp'].interpolate(method='linear')

In [90]:
data

,traffic_volume,temp,is_holiday
2012-10-02 09:00:00,5545.0,288.28,0.0
2012-10-02 10:00:00,4516.0,289.36,0.0
2012-10-02 11:00:00,4767.0,289.58,0.0
2012-10-02 12:00:00,5026.0,290.13,0.0
2012-10-02 13:00:00,4918.0,291.14,0.0
...,...,...,...
2018-09-30 19:00:00,3543.0,283.45,0.0
2018-09-30 20:00:00,2781.0,282.76,0.0
2018-09-30 21:00:00,2159.0,282.73,0.0
2018-09-30 22:00:00,1450.0,282.09,0.0


In [92]:
HORIZON = 24 * 7  # Прогноз на 1 неделю (168 часов)
TEST_SIZE = 24 * 14 # Откладываем 2 недели (336 часов)

# Разделение данных
train_full = data.iloc[:-TEST_SIZE]
test = data.iloc[-TEST_SIZE:]

# Оставляем для обучения только последний год данных
cutoff_date = train_full.index.max() - pd.Timedelta(days=365)
train = train_full.loc[train_full.index >= cutoff_date]

# Функция для генерации признаков из даты
def create_features(data):
    data = data.copy()
    data['hour'] = data.index.hour
    data['weekday'] = data.index.weekday
    data['month'] = data.index.month
    return data

train = create_features(train)
test = create_features(test)

In [94]:
test

,traffic_volume,temp,is_holiday,hour,weekday,month
2018-09-17 00:00:00,550.0,296.58,0.0,0,0,9
2018-09-17 01:00:00,280.0,296.19,0.0,1,0,9
2018-09-17 02:00:00,260.0,295.82,0.0,2,0,9
2018-09-17 03:00:00,344.0,295.83,0.0,3,0,9
2018-09-17 04:00:00,880.0,295.68,0.0,4,0,9
...,...,...,...,...,...,...
2018-09-30 19:00:00,3543.0,283.45,0.0,19,6,9
2018-09-30 20:00:00,2781.0,282.76,0.0,20,6,9
2018-09-30 21:00:00,2159.0,282.73,0.0,21,6,9
2018-09-30 22:00:00,1450.0,282.09,0.0,22,6,9
